In [1]:
%pwd


'/home/sheshpal/Documents/ml_llmops_project/Data_science[4]-end-to-end/DataSciecne_Project_forcasting/research'

In [2]:
import os

# Change directory to project root if currently in research directory
if os.getcwd().endswith("research"):
    os.chdir("../")

%pwd


'/home/sheshpal/Documents/ml_llmops_project/Data_science[4]-end-to-end/DataSciecne_Project_forcasting'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    train_data: Path
    store_data: Path


In [4]:
from src.forcastingproject.constants import *
from src.forcastingproject.utils.common import read_yaml_file, create_directories


In [7]:
from src.forcastingproject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH

class ComfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml_file(config_file_path)
        self.params = read_yaml_file(params_file_path)
        self.schema = read_yaml_file(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            train_data=Path(config.train_data),
            store_data=Path(config.store_data)
        )
        return data_ingestion_config




In [8]:
import os
import sys
import shutil
import pandas as pd
from src.forcastingproject.logging.logger import logger
from src.forcastingproject.exception.exception import CustomException


In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        """
        Copies raw train and store datasets into the data_ingestion artifacts directory.
        """
        try:
            logger.info("Starting data download/copy process...")
            raw_train_path = os.path.join(self.config.root_dir, "train.csv")
            raw_store_path = os.path.join(self.config.root_dir, "store.csv")

            if not os.path.exists(raw_train_path):
                shutil.copy(self.config.train_data, raw_train_path)
                logger.info(f"Copied {self.config.train_data} to {raw_train_path}")
            else:
                logger.info(f"File {raw_train_path} already exists.")

            if not os.path.exists(raw_store_path):
                shutil.copy(self.config.store_data, raw_store_path)
                logger.info(f"Copied {self.config.store_data} to {raw_store_path}")
            else:
                logger.info(f"File {raw_store_path} already exists.")

        except Exception as e:
            raise CustomException(e, sys)

    def merge_data(self) -> pd.DataFrame:
        """
        Merges train.csv and store.csv on Store column and saves as raw.csv in root_dir.
        """
        try:
            logger.info("Starting data merge process...")
            train_df = pd.read_csv(self.config.train_data, low_memory=False)
            store_df = pd.read_csv(self.config.store_data, low_memory=False)

            logger.info(f"Train data shape: {train_df.shape}")
            logger.info(f"Store data shape: {store_df.shape}")

            merged_df = pd.merge(train_df, store_df, on="Store", how="left")
            logger.info(f"Merged data shape: {merged_df.shape}")

            merged_data_path = os.path.join(self.config.root_dir, "raw.csv")
            merged_df.to_csv(merged_data_path, index=False)
            logger.info(f"Merged raw data saved at {merged_data_path}")

            return merged_df
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_data_ingestion(self):
        """
        Executes the full data ingestion pipeline: copies source files and merges them.
        """
        try:
            self.download_data()
            merged_df = self.merge_data()
            logger.info("Data Ingestion completed successfully.")
            return merged_df
        except Exception as e:
            raise CustomException(e, sys)


In [10]:
try:
    config_manager = ComfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.initiate_data_ingestion()
except Exception as e:
    raise e


[2026-11-14 21:22:12,174] forcastingProject INFO line 83: Created directory: artifacts
[2026-11-14 21:22:12,176] forcastingProject INFO line 83: Created directory: artifacts/data_ingestion
[2026-11-14 21:22:12,178] forcastingProject INFO line 10: Starting data download/copy process...
[2026-11-14 21:22:12,205] forcastingProject INFO line 16: Copied data/train.csv to artifacts/data_ingestion/train.csv
[2026-11-14 21:22:12,207] forcastingProject INFO line 22: Copied data/store.csv to artifacts/data_ingestion/store.csv
[2026-11-14 21:22:12,210] forcastingProject INFO line 34: Starting data merge process...
[2026-11-14 21:22:13,994] forcastingProject INFO line 38: Train data shape: (1017209, 9)
[2026-11-14 21:22:14,000] forcastingProject INFO line 39: Store data shape: (1115, 10)
[2026-11-14 21:22:14,429] forcastingProject INFO line 42: Merged data shape: (1017209, 18)
[2026-11-14 21:22:33,953] forcastingProject INFO line 46: Merged raw data saved at artifacts/data_ingestion/raw.csv
[2026-

In [11]:
# Verify ingested data
raw_data_path = os.path.join(data_ingestion_config.root_dir, "raw.csv")
if os.path.exists(raw_data_path):
    sample_df = pd.read_csv(raw_data_path, low_memory=False, nrows=5)
    print(f"Data ingested successfully! Shape: {pd.read_csv(raw_data_path, low_memory=False).shape}")
    display(sample_df.head())


Data ingested successfully! Shape: (1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN
